# 103 Spark - Movielens

The goal of this lab is to run some analysis on a different dataset, [MovieLens](https://grouplens.org/datasets/movielens/).

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/3.5.0/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.RDD.html)

**Download the dataset** from [here](https://big.csr.unibo.it/downloads/bigdata/ml-dataset.zip), unzip it and put it in the ```datasets/big``` folder.

- ml-movies.csv (<u>movieId</u>:Long, title:String, genres:String) 
    - genres are separated by pipelines  (e.g., "comedy|drama|action")
    - each movie is associated with many ratings

- ml-ratings.csv (<u>userId</u>:Long, <u>movieId</u>:Long, rating:Double, year:Int)
    - each rating is associated with many tags
    - ml-ratings-sample.csv is a small sample of ml-ratings.csv, useful for developing
- ml-tags.csv (<u>userId</u>:Long, <u>movieId</u>:Long, <u>tag</u>:String, year:Int) 

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[4]") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()
sc = spark.sparkContext

sc

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


<SparkContext master=local[4] appName=Local Spark>

In [2]:
path_to_datasets = "../../../../datasets/big/"

path_ml_movies = path_to_datasets + "ml-movies.csv"
path_ml_ratings = path_to_datasets + "ml-ratings-sample.csv"
path_ml_tags = path_to_datasets + "ml-tags.csv"

In [3]:
import re
from datetime import datetime, timezone
from typing import Optional, Tuple

class MovieLensParser:
    no_genres_listed = "(no genres listed)"
    comma_regex = re.compile(r',(?=(?:[^"]*"[^"]*")*[^"]*$)')
    pipe_regex = re.compile(r'\|(?=(?:[^"]*"[^"]*")*[^"]*$)')
    quotes = '"'

    @staticmethod
    def year_from_timestamp(timestamp: str) -> int:
        """Convert from timestamp (string) to year (int)."""
        ts = int(timestamp.strip())
        return datetime.fromtimestamp(ts, tz=timezone.utc).year

    @staticmethod
    def parse_movie_line(line: str) -> Optional[Tuple[int, str, str]]:
        """Parse a movie record into (movieId, title, genres)."""
        try:
            parts = MovieLensParser.comma_regex.split(line)
            title = parts[1].strip()
            if title.startswith(MovieLensParser.quotes):
                title = title[1:]
            if title.endswith(MovieLensParser.quotes):
                title = title[:-1]
            return (int(parts[0].strip()), title, parts[2].strip())
        except Exception:
            return None

    @staticmethod
    def parse_rating_line(line: str) -> Optional[Tuple[int, int, float, int]]:
        """Parse a rating record into (userId, movieId, rating, year)."""
        try:
            parts = MovieLensParser.comma_regex.split(line)
            year = MovieLensParser.year_from_timestamp(parts[3])
            return int(parts[0].strip()), int(parts[1].strip()), float(parts[2].strip()), year
        except Exception:
            return None

    @staticmethod
    def parse_tag_line(line: str) -> Optional[Tuple[int, int, str, int]]:
        """Parse a tag record into (userId, movieId, tag, year)."""
        try:
            parts = MovieLensParser.comma_regex.split(line)
            year = MovieLensParser.year_from_timestamp(parts[3])
            return (int(parts[0].strip()), int(parts[1].strip()), parts[2].strip(), year)
        except Exception:
            return None


In [4]:
rddMovies = sc.textFile(path_ml_movies).map(MovieLensParser.parse_movie_line).filter(lambda x: x is not None)
rddRatings = sc.textFile(path_ml_ratings).map(MovieLensParser.parse_rating_line).filter(lambda x: x is not None)
rddTags = sc.textFile(path_ml_tags).map(MovieLensParser.parse_tag_line).filter(lambda x: x is not None)

## 103-1 Datasets exploration

Cache the datasets and answer the following questions:

- How many (distinct) users, movies, ratings, and tags?
- How many (distinct) genres?
- On average, how many ratings per user?
- On average, how many ratings per movie?
- On average, how many genres per movie?
- What is the range of ratings?
- Which years? (print an ordered list)
- On average, how many ratings per year?

In [6]:
print("Example rddMovie:", rddMovies.first()) # (movieId, title, genres)
print("Example rddRatings:", rddRatings.first()) # (userId, movieId, rating, year)
print("Example rddTags:", rddTags.first()) # (userId, movieId, tag, year:Int)

Example rddMovie: (1, 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy')
Example rddRatings: (1, 307, 3.5, 2009)
Example rddTags: (14, 110, 'epic', 2015)


In [5]:
for (id, rdd) in sc._jsc.getPersistentRDDs().items():
    rdd.unpersist()

rddMoviesCached = rddMovies.cache()
rddRatingsCached = rddRatings.cache()
rddTagsCached = rddTags.cache()

nUsers = rddRatingsCached.map(lambda x: x[0]).distinct().count()
nMovies = rddMoviesCached.map(lambda x: x[0]).distinct().count()
print("Number of users:", nUsers)
print("Number of movies:", nMovies)
print("Number of ratings:", rddRatingsCached.count())
print("Number of tags:", rddTagsCached.map(lambda x: x[2]).distinct().count())

print("Number of genres:", rddMoviesCached.flatMap(lambda x: x[2].split("|")).distinct().count())

nRatings = rddRatingsCached.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b).map(lambda kv: kv[1]).sum()
print("Number of ratings per user:", nRatings / nUsers)

print("Number of ratings per movie:", nRatings / nMovies)

print("Number of genres per movie:", rddMoviesCached.map(lambda x: len(x[2].split("|"))).sum() / nMovies)

print(f"The range of ratings goes from {rddRatingsCached.map(lambda x: x[2]).min()} to {rddRatingsCached.map(lambda x: x[2]).max()}")

print("Ordered list of years:", sorted(rddRatingsCached.map(lambda x: x[3]).distinct().collect()))

# Computing the averege in another way just to try...
rddRatingsPerYear = rddRatingsCached. \
	map(lambda x: (x[3], 1)). \
	reduceByKey(lambda a, b: a + b). \
	map(lambda kv: (kv[1], 1)). \
	reduce(lambda x1, x2: (x1[0] + x2[0], x1[1] + x2[1]))
print("Number of ratings per year:", rddRatingsPerYear[0] / rddRatingsPerYear[1])

Number of users: 10073
Number of movies: 58098
Number of ratings: 1000000


Number of tags: 74702
Number of genres: 20
Number of ratings per user: 99.27529038022436
Number of ratings per movie: 17.212296464594306
Number of genres per movie: 1.8263451409687081
The range of ratings goes from 0.5 to 5.0
Ordered list of years: [1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]
Number of ratings per year: 43478.260869565216


## 103-2 Compute the average rating for each movie

- Export the result to a file
- Do not start from cached RDDs
- Evaluate:
  - Join-and-Aggregate
  - Aggregate-and-Join
  - Aggregate-and-BroadcastJoin

In [ ]:
path_output_avgRatPerMovie = "../../../../output/avgRatPerMovie"
# rdd.coalesce(1).toDF().write.format("csv").mode('overwrite').option("header", "true").save(path_output_avgRatPerMovie)

for (id, rdd) in sc._jsc.getPersistentRDDs().items():         
    rdd.unpersist()

### Join-and-Aggregate

In [ ]:
rddMoviesKV = rddMovies.map(lambda x: (x[0],x[1]))
avgRatPerMovie = rddRatings.\
    map(lambda x: (x[1],x[2])).\
    join(rddMoviesKV).\
    map(lambda x: ((x[0],x[1][1]),(x[1][0],1)) ).\
    reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])).\
    map(lambda x: (x[0][0], x[0][1], x[1][0]/x[1][1], x[1][1])).\
    coalesce(1).\
    toDF(["movieId", "title", "averageRating", "ratingCount"]).\
    write.format("csv").mode('overwrite').option("header", "true").save(path_output_avgRatPerMovie)

### Aggregate-and-Join

In [ ]:
rddMoviesKV = rddMovies.map(lambda x: (x[0],x[1]))
avgRatPerMovie = rddRatings.\
    map(lambda x: (x[1],(x[2],1))).\
    reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])).\
    mapValues(lambda x: (x[0]/x[1], x[1])).\
    join(rddMoviesKV).\
    map(lambda x: (x[0], x[1][1], x[1][0][0], x[1][0][1])).\
    coalesce(1).\
    toDF(["movieId", "title", "averageRating", "ratingCount"]).\
    write.format("csv").mode('overwrite').option("header", "true").save(path_output_avgRatPerMovie)

### Aggregate-and-BroadcastJoin

In [ ]:
rddMoviesKV = rddMovies.map(lambda x: (x[0],x[1]))
bRddMovies = sc.broadcast(rddMoviesKV.collectAsMap())
avgRatPerMovie = rddRatings.\
    map(lambda x: (x[1],(x[2],1))).\
    reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])).\
    mapValues(lambda x: (x[0]/x[1], x[1])).\
    map(lambda x: (x[0],bRddMovies.value.get(x[0]),x[1][0],x[1][1])).\
    coalesce(1).\
    toDF(["movieId", "title", "averageRating", "ratingCount"]).\
    write.format("csv").mode('overwrite').option("header", "true").save(path_output_avgRatPerMovie)

## 103-3 Compute the average rating for each genre

Two possible workflows:

1. Pre-aggregation (3 shuffles)

  - Aggregate ratings by movieId
  - Join with movies and map to genres
  - Aggregate by genres
  
2. Join & aggregate (2 shuffles)

  - Join with movies and map to genres
  - Aggregate by genres

In [ ]:
path_output_avgRatPerMovie = "../../../../output/avgRatPerGenre"

for (id, rdd) in sc._jsc.getPersistentRDDs().items():         
    rdd.unpersist()

## Exercise 

- Jobs
    - Calculate average rating by tag, and classify tags by result
        - Positive if >= 4
        - Negative if <= 2
        - Neutral otherwise
    - (optional) Count, for each movie, how many psotive/neutral/negative tags it received

- Implementation
    - Try it locally on the notebook
    - Add it as a new job to MovieLens in Lab04
    - Deploy it on AWS
        - Launch the AWS accademy
        - Create a cluster with 3-4 machines
        - Upload the jar
        - Launch the job via spark-submit by settings the executor configuration parameters

In [7]:
print("Example rddMovie:", rddMovies.first()) # (movieId, title, genres)
print("Example rddRatings:", rddRatings.first()) # (userId, movieId, rating, year)
print("Example rddTags:", rddTags.first()) # (userId, movieId, tag, year:Int)

Example rddMovie: (1, 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy')
Example rddRatings: (1, 307, 3.5, 2009)
Example rddTags: (14, 110, 'epic', 2015)


In [13]:
rddRatingsKV = rddRatings.\
    map(lambda x: (x[1],x[2]))

# map ->            (movieId, multiple genres)
# flatMapValues ->  (movieId, single genre)

rddTags.\
    map(lambda x: (x[1], x[2])).\
    join(rddRatingsKV).\
    map(lambda x: (x[1][0], (x[1][1], 1))). \
    reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])). \
    map(lambda x: (x[0], x[1][0] / x[1][1])). \
    map(lambda x: (x[0], "Positive" if x[1] >= 4 else ("Negative" if x[1] <= 2 else "Neutral"))). \
    take(10)
    # map(lambda x: (x[1][1],(x[1][0][0],x[1][0][1]))).\
    # reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])).\
    # map(lambda x: (x[0], x[1][0]/x[1][1], x[1][1])).\
    # coalesce(1).\
    # toDF(["genre", "averageRating", "genre_count"]).\
    # write.format("csv").mode('overwrite').option("header", "true").save(path_output_avgRatPerGenre)

[('sci-fi', 'Neutral'),
 ('must see', 'Positive'),
 ('orca', 'Positive'),
 ('space', 'Neutral'),
 ('trilogy', 'Positive'),
 ('blockbuster', 'Positive'),
 ('great story', 'Positive'),
 ("haven't seen it", 'Positive'),
 ('future', 'Neutral'),
 ('critically acclaimed', 'Positive')]